# Stage 2: Adaptive Observation Scheduling Engine

**Research Question:** Can an AI-driven scheduler optimize exoplanet observation allocation more effectively than static prioritization approaches?

---

## Architecture

```
Stage 1 Priority Scores + Uncertainty
        |
        v
[A] Observation Constraint Engine   <- AR(1) weather, time budget, visibility
        |
        v
[B] Scientific Gain Calculator      <- Gain = a*U + b*P + g*D  (b decays over time)
        |
        v
[C] 5 Scheduler Algorithms
    1. Static Priority              (Baseline 1)
    2. Detectability Greedy         (Baseline 2)
    3. Uncertainty Greedy           (Baseline 3)
    4. Adaptive Scheduler           (Our method)
    5. Oracle Scheduler             (Upper bound)
        |
        v
[D] Observation Simulator           <- SNR/weather/detectability-dependent noise
        |
        v
[E] Adaptive Reprioritization Loop  <- 30 rounds x 10 planets
        |
        v
[F] Evaluation & Metrics            <- 7 metrics including Campaign Diversity
        |
        v
[G] Streamlit Dashboard             <- see dashboard/app.py
```

## Simulation Parameters
| Parameter | Value |
|-----------|-------|
| Rounds | 30 |
| Planets per round | 10 |
| Telescope hours/night | 8 hrs |
| Weather model | AR(1), rho=0.65 |
| Exploration decay tau | 15 rounds |


In [1]:
# ==============================================================
# CELL 1 - PULL FROM GITHUB  (run at START of every session)
# ==============================================================
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

GITHUB_USER = 'rushikesh-D69'
REPO_NAME   = 'water'
BRANCH      = 'main'
REPO_URL    = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'

if IN_COLAB:
    REPO_PATH = f'/content/{REPO_NAME}'
    if not os.path.exists(REPO_PATH):
        print(f'[Git] Cloning {REPO_URL} ...')
        os.system(f'git clone {REPO_URL} {REPO_PATH}')
    else:
        print('[Git] Repo exists. Pulling latest ...')
        os.system(f'git -C {REPO_PATH} pull origin {BRANCH}')
    os.chdir(REPO_PATH)
    sys.path.insert(0, REPO_PATH)
    print('[Colab] Installing dependencies ...')
    os.system('pip install -q xgboost lightgbm shap scipy scikit-learn matplotlib seaborn requests joblib')
    print('[Colab] Ready.')
else:
    ROOT = os.path.abspath('.')
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)
    print(f'[Local] Project root: {ROOT}')


[Local] Project root: D:\PEOJECTS\water


---
## 1. Imports & Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Stage 1 pipeline
from src.data_acquisition import run_pipeline, ML_FEATURES, TARGET

# Stage 2 modules
from src.constraint_engine  import ObservationConstraintEngine, WeatherModel
from src.observation_simulator import ObservationSimulator
from src.scheduler import (
    StaticPriorityScheduler,
    DetectabilityGreedyScheduler,
    UncertaintyGreedyScheduler,
    AdaptiveScheduler,
    OracleScheduler,
    run_campaign,
)
from src.evaluation import run_full_evaluation

# Stage 1 ML pipeline for predictions + uncertainty
from src.ml_pipeline import run_ml_pipeline

print('[Setup] All imports OK.')


[Setup] All imports OK.


---
## 2. Load Stage 1 Data & Predictions

In [3]:
import joblib
from pathlib import Path

DATA_DIR   = Path('data')
MODELS_DIR = Path('models')

# Load processed dataset
df_ml = pd.read_csv(DATA_DIR / 'exoplanets_processed.csv')
print(f'[Data] Loaded {len(df_ml):,} planets | {len(df_ml.columns)} columns')

# Load Stage 1 models for predictions + uncertainty
available_models = list(MODELS_DIR.glob('*.joblib')) + list(MODELS_DIR.glob('*.json'))
print(f'[Models] Found: {[m.name for m in available_models]}')

# Load LightGBM (best Stage 1 model) for predictions
try:
    import lightgbm as lgb
    lgb_model = lgb.Booster(model_file=str(MODELS_DIR / 'lightgbm.joblib'))
    print('[Models] LightGBM loaded')
except Exception as e:
    print(f'[Models] LightGBM load failed: {e} - will rerun ML pipeline')
    lgb_model = None

# Load Random Forest for uncertainty (tree variance)
try:
    rf_model = joblib.load(MODELS_DIR / 'random_forest.joblib')
    print('[Models] Random Forest loaded (for uncertainty)')
except Exception as e:
    print(f'[Models] RF load failed: {e}')
    rf_model = None


[Data] Loaded 5,522 planets | 47 columns
[Models] Found: ['gradient_boosting.joblib', 'lightgbm.joblib', 'random_forest.joblib', 'xgboost_model.json']
[Models] LightGBM load failed: No module named 'lightgbm' - will rerun ML pipeline


[Models] Random Forest loaded (for uncertainty)

In [4]:
# Get ML feature columns (leakage-free)
FEATURE_COLS = [c for c in ML_FEATURES if c in df_ml.columns]
X = df_ml[FEATURE_COLS].fillna(df_ml[FEATURE_COLS].median()).values

# Predictions from LightGBM (or Random Forest as fallback)
if lgb_model is not None:
    mu_pred = lgb_model.predict(X)
elif rf_model is not None:
    mu_pred = rf_model.predict(X)
else:
    # Fallback: use priority_score directly
    mu_pred = df_ml[TARGET].values
    print('[Warn] Using raw priority_score as mu_pred')

mu_pred = np.clip(mu_pred, 0, 1)

# Uncertainty from Random Forest tree variance
if rf_model is not None:
    tree_preds = np.stack([tree.predict(X) for tree in rf_model.estimators_], axis=0)
    sigma_pred = np.std(tree_preds, axis=0)
else:
    # Fallback: uniform uncertainty
    sigma_pred = np.full(len(df_ml), 0.1)
    print('[Warn] Using uniform sigma=0.1')

sigma_pred = np.clip(sigma_pred, 0.01, 1.0)

# True priority scores (for Oracle scheduler)
true_priorities = df_ml[TARGET].values

print(f'[Predictions] mu:    mean={mu_pred.mean():.4f}  std={mu_pred.std():.4f}')
print(f'[Predictions] sigma: mean={sigma_pred.mean():.4f}  max={sigma_pred.max():.4f}')
print(f'[Oracle]      true:  mean={true_priorities.mean():.4f}  std={true_priorities.std():.4f}')


[Predictions] mu:    mean=0.5364  std=0.2557
[Predictions] sigma: mean=0.0637  max=0.3314
[Oracle]      true:  mean=0.5356  std=0.2717


---
## 3. Initialize Constraint Engine & Observation Simulator

In [5]:
# Simulation parameters
N_ROUNDS    = 30
K_PER_ROUND = 10
SEED        = 42

print(f'[Config] {N_ROUNDS} rounds x {K_PER_ROUND} planets = {N_ROUNDS * K_PER_ROUND} total observations')
print(f'         from {len(df_ml):,} candidate planets ({N_ROUNDS * K_PER_ROUND / len(df_ml) * 100:.1f}% of pool)')
print(f'         Telescope budget: 8 hrs/night | Weather AR(1) rho=0.65')


[Config] 30 rounds x 10 planets = 300 total observations
         from 5,522 candidate planets (5.4% of pool)
         Telescope budget: 8 hrs/night | Weather AR(1) rho=0.65


---
## 4. Run Observation Campaigns (All 5 Schedulers)

In [6]:
import copy

def make_fresh_simulator(seed_offset=0):
    return ObservationSimulator(
        df             = df_ml,
        initial_means  = mu_pred.copy(),
        initial_sigmas = sigma_pred.copy(),
        seed           = SEED + seed_offset,
    )

def make_fresh_ce(seed_offset=0):
    return ObservationConstraintEngine(df_ml, seed=SEED + seed_offset)

results = {}

# 1. Static Priority
print('\n[1/5] Static Priority Scheduler ...')
sim1 = make_fresh_simulator(1)
ce1  = make_fresh_ce(1)
s1   = StaticPriorityScheduler(df_ml, true_priorities)  # uses static scores
results['Static Priority'] = run_campaign(s1, sim1, ce1, N_ROUNDS, K_PER_ROUND, verbose=True)

# 2. Detectability Greedy
print('\n[2/5] Detectability Greedy Scheduler ...')
sim2 = make_fresh_simulator(2)
ce2  = make_fresh_ce(2)
s2   = DetectabilityGreedyScheduler('Detectability Greedy', df_ml)
results['Detectability Greedy'] = run_campaign(s2, sim2, ce2, N_ROUNDS, K_PER_ROUND, verbose=True)

# 3. Uncertainty Greedy
print('\n[3/5] Uncertainty Greedy Scheduler ...')
sim3 = make_fresh_simulator(3)
ce3  = make_fresh_ce(3)
s3   = UncertaintyGreedyScheduler('Uncertainty Greedy', df_ml)
results['Uncertainty Greedy'] = run_campaign(s3, sim3, ce3, N_ROUNDS, K_PER_ROUND, verbose=True)

# 4. Adaptive Scheduler (Our Method)
print('\n[4/5] Adaptive Scheduler (Our Method) ...')
sim4 = make_fresh_simulator(4)
ce4  = make_fresh_ce(4)
s4   = AdaptiveScheduler(df_ml, alpha_0=0.50, beta_0=0.30, gamma=0.20, tau=15.0)
results['Adaptive Scheduler'] = run_campaign(s4, sim4, ce4, N_ROUNDS, K_PER_ROUND, verbose=True)

# 5. Oracle Scheduler (Upper Bound)
print('\n[5/5] Oracle Scheduler (Upper Bound) ...')
sim5 = make_fresh_simulator(5)
ce5  = make_fresh_ce(5)
s5   = OracleScheduler(df_ml, true_priorities, alpha_0=0.50, beta_0=0.30, gamma=0.20, tau=15.0)
results['Oracle'] = run_campaign(s5, sim5, ce5, N_ROUNDS, K_PER_ROUND, verbose=True)

print('\n[Campaigns] All 5 schedulers complete.')
oracle_cum_gain = results['Oracle']['cumulative_gain']
print(f'[Oracle] Upper bound cumulative gain: {oracle_cum_gain:.4f}')



[1/5] Static Priority Scheduler ...

  Campaign: Static Priority
  30 rounds x 10 planets/round
  Round  5 | weather=Excellent (0.85)   | cum_gain=0.2930 | mean_prio=0.9191 | observed=28
  Round 10 | weather=Good (0.68)        | cum_gain=1.0762 | mean_prio=0.8301 | observed=57
  Round 15 | weather=Good (0.70)        | cum_gain=1.9359 | mean_prio=0.8631 | observed=88
  Round 20 | weather=Good (0.70)        | cum_gain=2.4794 | mean_prio=0.7661 | observed=114
  Round 25 | weather=Fair (0.62)        | cum_gain=2.9820 | mean_prio=0.8802 | observed=140


  Round 30 | weather=Good (0.72)        | cum_gain=3.9961 | mean_prio=0.8666 | observed=174

  Final: observed=174 planets | cum_gain=3.9961 | total_hrs=228.8

[2/5] Detectability Greedy Scheduler ...

  Campaign: Detectability Greedy
  30 rounds x 10 planets/round
  Round  5 | weather=Good (0.80)        | cum_gain=2.0402 | mean_prio=0.6457 | observed=36
  Round 10 | weather=Good (0.71)        | cum_gain=3.3590 | mean_prio=0.6141 | observed=67
  Round 15 | weather=Good (0.68)        | cum_gain=4.2203 | mean_prio=0.6685 | observed=96
  Round 20 | weather=Fair (0.54)        | cum_gain=5.0380 | mean_prio=0.6797 | observed=126
  Round 25 | weather=Good (0.79)        | cum_gain=5.7331 | mean_prio=0.5950 | observed=157
  Round 30 | weather=Good (0.71)        | cum_gain=6.4366 | mean_prio=0.7077 | observed=188

  Final: observed=188 planets | cum_gain=6.4366 | total_hrs=231.8

[3/5] Uncertainty Greedy Scheduler ...

  Campaign: Uncertainty Greedy
  30 rounds x 10 planets/round
  Round  5 | we

  Round 10 | weather=Fair (0.62)        | cum_gain=0.9521 | mean_prio=0.6364 | observed=46
  Round 15 | weather=Fair (0.54)        | cum_gain=1.2308 | mean_prio=0.5686 | observed=66
  Round 20 | weather=Good (0.79)        | cum_gain=1.8118 | mean_prio=0.4770 | observed=91
  Round 25 | weather=Good (0.78)        | cum_gain=2.3178 | mean_prio=0.4481 | observed=116
  Round 30 | weather=Good (0.78)        | cum_gain=3.0187 | mean_prio=0.6815 | observed=143

  Final: observed=143 planets | cum_gain=3.0187 | total_hrs=228.8

[4/5] Adaptive Scheduler (Our Method) ...

  Campaign: Adaptive Scheduler
  30 rounds x 10 planets/round
  Round  5 | weather=Good (0.73)        | cum_gain=1.1999 | mean_prio=0.8994 | observed=45
  Round 10 | weather=Good (0.81)        | cum_gain=2.1214 | mean_prio=0.7513 | observed=86
  Round 15 | weather=Good (0.76)        | cum_gain=3.1342 | mean_prio=0.6877 | observed=124
  Round 20 | weather=Good (0.74)        | cum_gain=4.1304 | mean_prio=0.6365 | observed=162


  Round 25 | weather=Excellent (0.90)   | cum_gain=4.9552 | mean_prio=0.6121 | observed=200
  Round 30 | weather=Good (0.74)        | cum_gain=5.7872 | mean_prio=0.7269 | observed=237

  Final: observed=237 planets | cum_gain=5.7872 | total_hrs=228.0

[5/5] Oracle Scheduler (Upper Bound) ...

  Campaign: Oracle
  30 rounds x 10 planets/round
  Round  5 | weather=Good (0.71)        | cum_gain=1.1615 | mean_prio=0.9322 | observed=45
  Round 10 | weather=Good (0.80)        | cum_gain=2.1053 | mean_prio=0.7695 | observed=85
  Round 15 | weather=Good (0.83)        | cum_gain=3.0133 | mean_prio=0.7801 | observed=124
  Round 20 | weather=Good (0.78)        | cum_gain=4.1431 | mean_prio=0.6404 | observed=162
  Round 25 | weather=Good (0.76)        | cum_gain=4.9699 | mean_prio=0.6240 | observed=198
  Round 30 | weather=Good (0.66)        | cum_gain=5.7391 | mean_prio=0.7530 | observed=236

  Final: observed=236 planets | cum_gain=5.7391 | total_hrs=227.0

[Campaigns] All 5 schedulers complete.

---
## 5. Evaluation: 7 Metrics + 7 Plots

In [7]:
# Get weather history from one of the campaigns
weather_history = results.get('Adaptive Scheduler', {}).get('weather_history', [])

comparison_df = run_full_evaluation(
    results         = results,
    n_rounds        = N_ROUNDS,
    k_per_round     = K_PER_ROUND,
    n_planets       = len(df_ml),
    weather_history = weather_history,
    df              = df_ml,
    oracle_cum_gain = oracle_cum_gain,
)

print('\nFull Comparison Table:')
display(comparison_df)



[Eval] Computing metrics ...

[Eval] Generating plots ...


[Plot] Saved -> D:\PEOJECTS\water\plots\s2_cumulative_gain.png


[Plot] Saved -> D:\PEOJECTS\water\plots\s2_uncertainty_evolution.png
[Plot] Saved -> D:\PEOJECTS\water\plots\s2_weight_decay.png


[Plot] Saved -> D:\PEOJECTS\water\plots\s2_regret.png


[Plot] Saved -> D:\PEOJECTS\water\plots\s2_efficiency.png
[Plot] Saved -> D:\PEOJECTS\water\plots\s2_weather_sequence.png


[Plot] Saved -> D:\PEOJECTS\water\plots\s2_diversity.png


[Plot] Saved -> D:\PEOJECTS\water\plots\s2_pareto_frontier.png

[Eval] Scheduler Comparison:
 Rank            Scheduler  Composite Score  Cum. Sci. Gain  Regret vs Oracle  Diversity Score  Priority Coverage  Telescope Utilization  Obs. Efficiency  Mean sigma Reduction  Planets Observed  Total Hrs Used
    1               Oracle           1.0000          5.7391            0.0000           0.6003             0.7746                  0.946           0.0253                0.0204               236           227.0
    2   Adaptive Scheduler           0.9987          5.7872            0.0000           0.5992             0.7713                  0.950           0.0254                0.0200               237           228.0
    3 Detectability Greedy           0.9555          6.4366            0.0000           0.5537             0.6776                  0.966           0.0277                0.0179               188           231.8
    4      Static Priority           0.8040          3.9961        

,Rank,Scheduler,Composite Score,Cum. Sci. Gain,Regret vs Oracle,Diversity Score,Priority Coverage,Telescope Utilization,Obs. Efficiency,Mean sigma Reduction,Planets Observed,Total Hrs Used
0,1,Oracle,1.0000,5.7391,0.0000,0.6003,0.7746,0.946,0.0253,0.0204,236,227.0
1,2,Adaptive Scheduler,0.9987,5.7872,0.0000,0.5992,0.7713,0.950,0.0254,0.0200,237,228.0
2,3,Detectability Greedy,0.9555,6.4366,0.0000,0.5537,0.6776,0.966,0.0277,0.0179,188,231.8
3,4,Static Priority,0.8040,3.9961,0.3037,0.5344,0.9854,0.953,0.0174,0.0630,174,228.8
4,5,Uncertainty Greedy,0.6605,3.0187,0.4740,0.4767,0.6723,0.953,0.0132,0.0989,143,228.8


---
## 6. Adaptive Scheduler Deep-Dive

In [8]:
adaptive_logs = results['Adaptive Scheduler']['logs_df']
oracle_logs   = results['Oracle']['logs_df']
adaptive_obs  = results['Adaptive Scheduler']['obs_history_df']

print('=== Adaptive Scheduler - Round-by-Round Summary ===')
display(adaptive_logs[['round','weather','time_used_hrs','alpha_t','beta_t',
                        'cum_sci_gain','mean_priority','mean_sigma_before',
                        'mean_sigma_after','top_target']].head(30))


=== Adaptive Scheduler - Round-by-Round Summary ===


,round,weather,time_used_hrs,alpha_t,beta_t,cum_sci_gain,mean_priority,mean_sigma_before,mean_sigma_after,top_target
0,1,0.854896,7.34,0.5099,0.2862,0.3696,0.906075,0.069466,0.069466,HD 219134 b
1,2,0.771710,7.74,0.5195,0.2728,0.6676,0.912827,0.061693,0.061693,pi Men c
2,3,0.736477,7.83,0.5288,0.2597,0.9327,0.901899,0.053698,0.053698,HD 63433 b
3,4,0.772456,7.67,0.5378,0.2471,1.0491,0.947264,0.026217,0.026217,BD+05 4868 A b
4,5,0.731942,7.65,0.5465,0.2349,1.1999,0.899358,0.045389,0.045389,GJ 876 d
5,6,0.770262,8.00,0.5549,0.2232,1.3660,0.893578,0.049193,0.049193,GJ 581 e
6,7,0.750074,7.31,0.5630,0.2118,1.5927,0.866319,0.069013,0.069013,TOI-5789 c
7,8,0.628545,7.23,0.5708,0.2009,1.7377,0.855449,0.046051,0.046051,HR 858 c
8,9,0.739539,7.23,0.5783,0.1904,1.8669,0.865155,0.047894,0.047894,GJ 806 c
9,10,0.805448,7.61,0.5855,0.1804,2.1214,0.751331,0.085382,0.085382,GJ 581 b


In [9]:
print('=== Top 20 Observations by Scientific Gain (Adaptive) ===')
top_obs = (adaptive_obs
    .assign(sci_gain=lambda d: d['sigma_before'] * d['detectability'])
    .sort_values('sci_gain', ascending=False)
    .head(20)[['round','planet_name','mu_before','sigma_before','mu_after',
               'sigma_after','weather','snr_effective','sci_gain']]
)
display(top_obs)


=== Top 20 Observations by Scientific Gain (Adaptive) ===


,round,planet_name,mu_before,sigma_before,mu_after,sigma_after,weather,snr_effective,sci_gain
134,17,HD 63433 c,0.706031,0.139452,0.730191,0.117813,0.744,0.550611,0.103204
1,1,55 Cnc e,0.871121,0.126004,0.992106,0.110488,0.855,0.453323,0.096350
100,12,AU Mic c,0.753599,0.106002,0.803553,0.092498,0.818,0.631911,0.081887
3,1,GJ 436 b,0.716442,0.103171,0.767703,0.090874,0.855,0.662223,0.079909
54,7,TOI-5789 c,0.675968,0.113330,0.642075,0.093641,0.750,0.318347,0.078274
13,2,AU Mic b,0.700424,0.099510,0.619040,0.085940,0.772,0.596822,0.076929
123,15,WASP-107 b,0.623241,0.115354,0.672044,0.094500,0.764,0.446999,0.076831
27,4,BD+05 4868 A b,0.846688,0.108544,0.892423,0.090525,0.772,0.503254,0.075871
9,2,pi Men c,0.902575,0.107620,0.972772,0.089408,0.772,0.324155,0.074358
18,3,HD 63433 b,0.878227,0.106377,0.852929,0.086814,0.736,0.294771,0.071510


---
## 7. Regret Analysis vs Oracle

In [10]:
from src.evaluation import compute_regret

print('=== Final Regret vs Oracle ===')
for name, res in results.items():
    if name == 'Oracle':
        continue
    logs = res['logs_df']
    if logs.empty:
        continue
    regret_series = compute_regret(logs, oracle_cum_gain)
    final_regret  = regret_series.iloc[-1]
    avg_regret    = regret_series.mean()
    print(f'  {name:<25} | Final regret: {final_regret:.4f} | Avg regret: {avg_regret:.4f}')


=== Final Regret vs Oracle ===
  Static Priority           | Final regret: 0.3037 | Avg regret: 0.6798
  Detectability Greedy      | Final regret: 0.0000 | Avg regret: 0.3004
  Uncertainty Greedy        | Final regret: 0.4740 | Avg regret: 0.7450
  Adaptive Scheduler        | Final regret: 0.0000 | Avg regret: 0.4461


---
## 8. Campaign Diversity Analysis

In [11]:
from src.evaluation import compute_campaign_diversity

print('=== Campaign Diversity Scores ===')
for name, res in results.items():
    obs = res['obs_history_df']
    if obs.empty or 'planet_idx' not in obs.columns:
        continue
    obs_idx = obs['planet_idx'].unique().tolist()
    div = compute_campaign_diversity(obs_idx, df_ml)
    print(f'\n  {name}:')
    for k, v in div.items():
        print(f'    {k:<30}: {v:.4f}')


=== Campaign Diversity Scores ===

  Static Priority:
    stellar_type_entropy          : 0.6631
    temperature_diversity         : 0.0833
    orbital_diversity             : 0.3388
    mass_diversity                : 0.5868
    distance_coverage             : 1.0000
    diversity_score               : 0.5344

  Detectability Greedy:
    stellar_type_entropy          : 0.7456
    temperature_diversity         : 0.1526
    orbital_diversity             : 0.4522
    mass_diversity                : 0.4879
    distance_coverage             : 0.9299
    diversity_score               : 0.5537

  Uncertainty Greedy:
    stellar_type_entropy          : 0.6560
    temperature_diversity         : 0.1136
    orbital_diversity             : 0.2515
    mass_diversity                : 0.3622
    distance_coverage             : 1.0000
    diversity_score               : 0.4767

  Adaptive Scheduler:
    stellar_type_entropy          : 0.6953
    temperature_diversity         : 0.1773
    orbital_div

---
## 9. Save Results

In [12]:
from pathlib import Path
DATA_DIR = Path('data')

# Save comparison table
comparison_df.to_csv(DATA_DIR / 'stage2_comparison.csv', index=False)
print(f'[Save] Comparison table -> data/stage2_comparison.csv')

# Save per-scheduler logs
for name, res in results.items():
    safe_name = name.lower().replace(' ', '_')
    if not res['logs_df'].empty:
        res['logs_df'].to_csv(DATA_DIR / f's2_{safe_name}_logs.csv', index=False)
        print(f'[Save] {name} logs -> data/s2_{safe_name}_logs.csv')

# Save adaptive observation history
adap_obs = results['Adaptive Scheduler']['obs_history_df']
if not adap_obs.empty:
    adap_obs.to_csv(DATA_DIR / 's2_adaptive_obs_history.csv', index=False)
    print('[Save] Adaptive obs history -> data/s2_adaptive_obs_history.csv')

print('\n[Done] Stage 2 complete.')


[Save] Comparison table -> data/stage2_comparison.csv
[Save] Static Priority logs -> data/s2_static_priority_logs.csv
[Save] Detectability Greedy logs -> data/s2_detectability_greedy_logs.csv
[Save] Uncertainty Greedy logs -> data/s2_uncertainty_greedy_logs.csv
[Save] Adaptive Scheduler logs -> data/s2_adaptive_scheduler_logs.csv
[Save] Oracle logs -> data/s2_oracle_logs.csv
[Save] Adaptive obs history -> data/s2_adaptive_obs_history.csv

[Done] Stage 2 complete.


---
## Push to GitHub

In [13]:
# ==============================================================
# LAST CELL - PUSH TO GITHUB  (run at END of every session)
# ==============================================================
import os

IN_COLAB = 'google.colab' in sys.modules

GITHUB_USER = 'rushikesh-D69'
REPO_NAME   = 'water'
BRANCH      = 'main'
GIT_EMAIL   = 'rikki0501hanuman@gmail.com'
GIT_NAME    = 'rushikesh-D69'

if IN_COLAB:
    os.system(f'git config user.email "{GIT_EMAIL}"')
    os.system(f'git config user.name "{GIT_NAME}"')
    print('[Git] Pulling latest before push ...')
    os.system(f'git pull origin {BRANCH}')
    print('[Git] Staging all files ...')
    os.system('git add -A')
    status = os.popen('git status --porcelain').read().strip()
    if status:
        print('[Git] Changes detected:')
        print(status)
        os.system('git commit -m "Colab sync: Stage 2 results, plots, data"')
        from google.colab import userdata
        token  = userdata.get('GITHUB_TOKEN')
        remote = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
        os.system(f'git remote set-url origin {remote}')
        print('[Git] Pushing to GitHub ...')
        os.system(f'git push origin {BRANCH}')
        print('[Git] Push complete.')
    else:
        print('[Git] No changes to push.')
else:
    print('[Local] Run: git add -A && git commit -m "msg" && git push')


[Local] Run: git add -A && git commit -m "msg" && git push
